# IRIS-X ARC-AGI-2 Trial 002 repair smoke 002
Private, network-disabled, competition-mounted L4 x4 smoke authorized by Decision 0015. Exactly four distinct tasks each receive one deterministic Qwen3.5-4B generation invocation; no TTT, retry, warm-up generation, or competition submission is performed.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import stat
import subprocess
import sys
import zipfile
import torch

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
input_root = Path('/kaggle/input')
hardware = []
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    hardware.append({'index': index, 'name': props.name, 'memory_mib': int(props.total_memory // (1024 * 1024)), 'uuid': str(getattr(props, 'uuid', 'unknown'))})
smi = subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,uuid,pci.bus_id', '--format=csv,noheader,nounits'], capture_output=True, text=True, check=False, timeout=30)
notebook_preflight = {'run_id': 'trial-002-smoke-002', 'torch_cuda_device_count': torch.cuda.device_count(), 'gpus': hardware, 'nvidia_smi_exit_code': smi.returncode, 'nvidia_smi_rows': [line.strip() for line in smi.stdout.splitlines() if line.strip()]}
Path('/kaggle/working/notebook-preflight.json').write_text(json.dumps(notebook_preflight, sort_keys=True) + '\n', encoding='utf-8')
if torch.cuda.device_count() != 4 or len(hardware) != 4 or any('L4' not in item['name'] or item['memory_mib'] < 22000 for item in hardware) or smi.returncode != 0 or len(notebook_preflight['nvidia_smi_rows']) != 4:
    raise RuntimeError(f'four-L4 runtime preflight failed: {notebook_preflight}')
challenge_matches = [p for p in input_root.rglob('arc-agi_training_challenges.json') if 'arc-prize-2026-arc-agi-2' in p.as_posix()]
if len(challenge_matches) != 1:
    raise RuntimeError(f'ARC-AGI-2 competition challenge mount discovery failed: {challenge_matches}')
competition_challenges = challenge_matches[0]
outbound_matches = list(input_root.rglob('trial-002-smoke-002.zip'))
if not outbound_matches:
    extracted_matches = [p.parent for p in input_root.rglob('manifest.json') if p.parent.name == 'trial-002-smoke-002']
    if len(extracted_matches) != 1:
        raise RuntimeError(f'bound outbound discovery failed: archives={outbound_matches}, extracted={extracted_matches}')
    extracted = extracted_matches[0]
    outbound = Path('/kaggle/working/trial-002-smoke-002.zip')
    with zipfile.ZipFile(outbound, 'w', allowZip64=True) as archive:
        for source in [extracted / 'manifest.json', *sorted(p for p in extracted.rglob('*') if p.is_file() and p.name != 'manifest.json')]:
            logical = source.relative_to(extracted).as_posix()
            info = zipfile.ZipInfo(logical, (1980, 1, 1, 0, 0, 0))
            info.compress_type = zipfile.ZIP_STORED
            info.create_system = 3
            info.external_attr = (stat.S_IFREG | 0o644) << 16
            archive.writestr(info, source.read_bytes())
    outbound_matches = [outbound]
model_matches = [p.parent for p in input_root.rglob('config.json') if (p.parent / 'model.safetensors.index.json').is_file()]
wheel_matches = [p.parent for p in input_root.rglob('transformers-5.8.1-py3-none-any.whl')]
if len(outbound_matches) != 1 or len(model_matches) != 1 or len(wheel_matches) != 1:
    raise RuntimeError(f'bound input discovery failed: outbound={outbound_matches}, model={model_matches}, wheels={wheel_matches}')
outbound, model_root, wheel_root = outbound_matches[0], model_matches[0], wheel_matches[0]
EXPECTED_OUTBOUND_SHA256 = '__IRISX_OUTBOUND_SHA256__'
if hashlib.sha256(outbound.read_bytes()).hexdigest() != EXPECTED_OUTBOUND_SHA256:
    raise RuntimeError('reconstructed outbound bytes do not match the preflight-bound digest')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(wheel_root), '--upgrade', '--no-deps', 'irisx==0.2.0', 'transformers==5.8.1', 'jinja2==3.1.6'], check=True)
result = Path('/kaggle/working/result.zip')
checkpoint = Path('/kaggle/working/checkpoint.json')
progress = Path('/kaggle/working/progress.jsonl')
command = [sys.executable, '-m', 'irisx.kaggle_runner_trial002', '--outbound', str(outbound), '--model-root', str(model_root), '--competition-challenges', str(competition_challenges), '--output', str(result), '--checkpoint-output', str(checkpoint), '--progress-output', str(progress)]
subprocess.run(command, check=True)
print('result:', result, result.stat().st_size)
print('checkpoint:', checkpoint, checkpoint.stat().st_size)
print('progress:', progress, progress.stat().st_size)
